# 20 — Sequential action components and environment-step returns

Control [Gymnasium Reacher-v5](https://gymnasium.farama.org/environments/mujoco/reacher/),
a two-joint arm reaching toward a target. Choose the first joint torque, then the second
conditioned on the first, and only then advance the environment with `[torque_0, torque_1]`.
One shared head emits 33 choices at each of two readout tokens, instead of 33² joint choices.
Both torques lie in `[-1, 1]`; they act on the same arm and jointly determine its motion.
This is a small executable wiring and training example, not a trained reaching agent.

Install the repository's `examples` extra, which includes `gymnasium[mujoco]`.
The example runs on CPU without rendering, so no display or model downloads are needed.

Each component choice is a separate **decision row**, with one readout token. The transformer
sees the earlier choices through causal history. The physical environment advances once
per completed vector; the two clocks are deliberately separate.

Set `BACKUP_MODE` to:
- `"next_action"`: ordinary one-step DQN, bootstrapping from the next component.
- `"n_actions"`: `nstep_gate(n=ACTION_COMPONENTS)`, reaching the same component in the next environment step.
- `"next_step"`: `nstep_gate(n=1, step_key="environment_step")`, reaching the next environment boundary from every component.

Internal rows have reward 0, discount 1, and no done flags. Boundary rows carry Reacher's
reward, physical observations, and discount γ. The marker describes the transition **into**
the row, as do action IDs and rewards. N-step targets use recorded intermediate choices:
they evaluate those sampled completions, not an exact maximum over all possible suffixes.


In [ ]:
from copy import deepcopy

import gymnasium as gym
import numpy as np
from tokenizers import Tokenizer as TextTokenizer
from tokenizers.models import WordLevel
import torch
from transformers import PreTrainedTokenizerFast

from mouse_core import AdamW, polyak_update
from mouse_core.data import DataLoader, Datastore, Tokenizer, to_device
from mouse_core.models import (
    Model,
    DecodeCache,
    prepare_decode,
    Backbone,
    Embedding,
    MLP,
    DiscretizedActionHead,
    select_discrete_action,
)
from mouse_core.objectives import DqnObjective, boundary_discount, nstep_gate

# Environment
MAX_ACTIONS = 33
MAX_STEPS_PER_EPISODE = gym.spec("Reacher-v5").max_episode_steps  # native limit
BACKUP_MODE = "next_step"

# Model
HIDDEN_DIM = 32
BACKBONE_LAYERS = 1
BACKBONE_HEADS = 2
NUM_FREQUENCIES = 4
PROJECTION_LAYERS = 2
HEAD_LAYERS = 1
HEAD_SCALE = 0.1  # final prediction linear's initialization

# Collection and rollout
EPISODES_PER_CYCLE = 2
EPSILON = 0.2

# Training
NUM_CYCLES = 2
TRAIN_STEPS = 2

# Optimizer learning rates
LR_EMBEDDING = 3e-4  # token embeddings and numeric Fourier features
LR_BACKBONE = 3e-4  # backbone
LR_PROJECTION = 3e-4  # projection MLP and its input RMSNorm
LR_HEADS = 3e-4  # prediction heads and their input RMSNorms

# Target interpolation rates
POLYAK_TAU_EMBEDDING = 0.01  # delayed embedding
POLYAK_TAU_BACKBONE = 0.01  # delayed backbone
POLYAK_TAU_PROJECTION = 0.01  # delayed projection
POLYAK_TAU_VALUE = 0.01  # delayed value head
POLYAK_TAU_ACTION = 1.0  # independent action-selection head

# Random seeds
SEED = 7

# Runtime
device = torch.device("cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))
torch.manual_seed(SEED)
action_rng = np.random.default_rng(SEED)


## Reacher environment

Reacher has one continuous action vector with two joint torques and a ten-value
observation (joint angles encoded as sine/cosine, target position, joint velocities,
and fingertip displacement). Both torque selections happen before `env.step`;
internal choices cannot reveal a future physical observation. The environment supplies
one reward for the complete vector.

Use Reacher-v5's native limit of 50 physical steps (100 component decisions). The
environment rewards fingertip proximity throughout the episode and truncates at that
limit; reaching the target does not terminate it. Training keeps the bootstrap value
at time-limit truncation. Replay sizing reads the native limit instead of shortening it.


In [ ]:
env = gym.make("Reacher-v5")
assert env.action_space.shape == (2,)
assert env.observation_space.shape == (10,)
ACTION_COMPONENTS = env.action_space.shape[0]
OBSERVATION_DIM = env.observation_space.shape[0]
# Both joints share the same bounds, allowing one scalar codebook and Q head.
np.testing.assert_array_equal(env.action_space.low, [-1.0, -1.0])
np.testing.assert_array_equal(env.action_space.high, [1.0, 1.0])
action_vectors = torch.linspace(-1.0, 1.0, MAX_ACTIONS)[:, None]
assert all(env.action_space.contains(np.repeat(v.numpy(), ACTION_COMPONENTS))
           for v in action_vectors)


## Tokenization

`component` identifies the torque to choose next. `torque` records the previous choice;
all physical observations remain unchanged between choices. Each row has one readout.
The category ID is only an objective label, while its physical torque enters the model.
The final observation is retained for bootstrapping; reset episodes have separate groups.


In [ ]:
vocabulary = {
    "[UNK]": 0, "torque": 1, "cos": 2, "sin": 3,
    "velocity": 4, "reward": 5, "reset": 6, "readout": 7,
}
vocabulary["component"] = len(vocabulary)
for j in range(OBSERVATION_DIM):
    vocabulary[f"obs{j}"] = len(vocabulary)
text_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=TextTokenizer(WordLevel(vocab=vocabulary, unk_token="[UNK]")),
    unk_token="[UNK]",
)

tokenizer = Tokenizer(
    tokenizer=text_tokenizer,
    input_fields=[
        {"type": "numeric", "input_field": "torque", "format": "torque",
         "fourier_min": -2.0, "fourier_max": 2.0},
        *[
            {"type": "numeric", "input_field": "observation", "input_index": j,
             "format": f"obs{j}", "fourier_min": -16.0, "fourier_max": 16.0}
            for j in range(OBSERVATION_DIM)
        ],
        {"type": "numeric", "input_field": "component", "format": "component",
         "fourier_min": -1.0, "fourier_max": float(ACTION_COMPONENTS + 1)},
        {"type": "numeric", "input_field": "reward", "format": "reward",
         "fourier_min": -32.0, "fourier_max": 32.0},
        {"type": "numeric", "input_field": "reset_frame", "format": "reset",
         "fourier_min": -2.0, "fourier_max": 2.0},
        {"type": "token", "format": "readout", "head_output": True},
    ],
    objective_fields=[
        {"input_field": "action_id", "output_field": "action"},
        {"input_field": "reward"},
        {"input_field": "terminated"},
        {"input_field": "truncated"},
        {"input_field": "environment_step"},
    ],
)

## Shared sequential Q head

The same head is used at each component; history and the component index distinguish its decisions.


In [ ]:
class Agent(Model):
    def __init__(self, *, embedding, backbone, projection, value_head):
        super().__init__()
        self.embedding = embedding
        self.backbone = backbone
        self.projection = projection
        self.value_head = value_head

    def get_action(self, *, out, temperature, num_actions=None):
        action_ids = select_discrete_action(
            scores=out["predictions"]["action_value"], valid=out["head_valid_mask"],
            temperature=temperature, num_actions=num_actions,
        )
        return action_ids

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to_tensors(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_token_ids"]
        head_step_ids = tensors["head_step_ids"]
        embed = self.embedding(
            token_ids, numeric_values=numeric_values, numeric_mask=numeric_mask,
        )
        head_valid_mask = None
        if use_cache:
            embed, lengths, head_token_ids, head_valid_mask = prepare_decode(
                embeds=embed, tokens=tokens, sequence_ids=sequence_ids,
                head_token_ids=head_token_ids, head_step_ids=head_step_ids,
            )
            if cache is None:
                cache = DecodeCache(sessions=(self.backbone.decode_session(batch_size=tokens.B),))
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape, features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        predictions = self.head(h=h)
        return {
            "predictions": predictions,
            "last_hidden_state": features,
            "head_token_ids": head_token_ids,
            "head_valid_mask": head_valid_mask,
            "cache": cache,
        }

    def head(self, *, h):
        h = self.projection(h.float())
        action_value_predictions = self.value_head(h)
        return {"action_value": action_value_predictions}


class DelayedAgent(Model):
    def __init__(self, *, embedding, backbone, projection, value_head, action_head):
        super().__init__()
        self.embedding = embedding
        self.backbone = backbone
        self.projection = projection
        self.value_head = value_head
        self.action_head = action_head

    forward = Agent.forward

    def head(self, *, h):
        h = self.projection(h.float())
        backup_value_predictions = self.value_head(h)
        backup_action_predictions = self.action_head(h)
        return {
            "backup_value": backup_value_predictions,
            "backup_action": backup_action_predictions,
        }

    def polyak(self, *, online, embedding_tau, backbone_tau, projection_tau, value_tau, action_tau):
        if not (
            0 <= embedding_tau <= 1
            and 0 <= backbone_tau <= 1
            and 0 <= projection_tau <= 1
            and 0 <= value_tau <= 1
            and 0 <= action_tau <= 1
        ):
            raise ValueError("Polyak rates must be in [0, 1].")
        polyak_update(target=self.embedding, source=online.embedding, tau=embedding_tau)
        polyak_update(target=self.backbone, source=online.backbone, tau=backbone_tau)
        polyak_update(target=self.projection, source=online.projection, tau=projection_tau)
        polyak_update(target=self.value_head, source=online.value_head, tau=value_tau)
        polyak_update(target=self.action_head, source=online.value_head, tau=action_tau)


def make_agent():
    embedding = Embedding(hidden_dim=HIDDEN_DIM, vocab_size=len(vocabulary), num_frequencies=NUM_FREQUENCIES)
    backbone = Backbone(
        architecture="qwen3",
        hidden_dim=HIDDEN_DIM,
        num_layers=BACKBONE_LAYERS,
        num_heads=BACKBONE_HEADS,
        train_kernel="reference",
        decode_kernel="flex",
        dtype=torch.float32,
    )

    projection = MLP(
        in_features=backbone.hidden_dim,
        out_features=backbone.hidden_dim,
        hidden_dim=backbone.hidden_dim,
        num_layers=PROJECTION_LAYERS,
    )

    head = DiscretizedActionHead(
        in_features=backbone.hidden_dim,
        action_vectors=action_vectors,
        hidden_dim=backbone.hidden_dim,
        num_layers=HEAD_LAYERS,
        scale=HEAD_SCALE,
        propagate_gradient=1.0,
    )


    agent = Agent(
        embedding=embedding,
        backbone=backbone,
        projection=projection,
        value_head=head,
    ).train().to(device)
    return agent

model = make_agent()

delayed_model = DelayedAgent(
    embedding=deepcopy(model.embedding).requires_grad_(False),
    backbone=deepcopy(model.backbone).requires_grad_(False),
    projection=deepcopy(model.projection).requires_grad_(False),
    value_head=deepcopy(model.value_head).requires_grad_(False),
    action_head=deepcopy(model.value_head).requires_grad_(False),
).eval()

head = model.value_head
print(f"{ACTION_COMPONENTS} sequential choices of {MAX_ACTIONS} torques")


## Collection

Internal decision rows keep the observation fixed; the completed vector advances Reacher once.


In [ ]:
def make_row(*, observation, episode_index, action_id=0, torque=0.0,
             component=0, reward=0.0, terminated=False, truncated=False,
             environment_step=False, reset_frame=False):
    return dict(observation=np.asarray(observation, dtype=np.float32).copy(),
                episode_index=int(episode_index), action_id=int(action_id), torque=float(torque),
                component=float(component), reward=float(reward), terminated=bool(terminated),
                truncated=bool(truncated), environment_step=bool(environment_step),
                reset_frame=float(reset_frame))


@torch.no_grad()
def collect_episode(*, episode_index, store):
    model.eval()
    observation, _ = env.reset(seed=SEED + episode_index)
    history = [make_row(observation=observation, episode_index=episode_index, reset_frame=True)]
    store.append(data=history[0])
    episode_return = 0.0
    while True:
        torques = []
        for component in range(ACTION_COMPONENTS):
            out = model(tokenizer.pack_rows(rows=[history], continuing=None))
            scores = out["predictions"]["action_value"][-1]
            action_id = (int(action_rng.integers(MAX_ACTIONS)) if action_rng.random() < EPSILON
                         else int(scores.argmax().item()))
            torque = float(head.discrete_to_continuous(action_id)[0])
            torques.append(torque)
            advanced = component == ACTION_COMPONENTS - 1
            reward, terminated, truncated = 0.0, False, False
            if advanced:
                continuous_action = np.asarray(torques, dtype=np.float32)
                assert env.action_space.contains(continuous_action)
                observation, reward, terminated, truncated, _ = env.step(continuous_action)
                episode_return += reward
            row = make_row(observation=observation, episode_index=episode_index,
                           action_id=action_id, torque=torque,
                           component=(component + 1) % ACTION_COMPONENTS,
                           reward=reward, terminated=terminated, truncated=truncated,
                           environment_step=advanced)
            history.append(row)
            store.append(data=row)
        if terminated or truncated:
            return episode_return, history


## Choose the backup horizon and train

Only environment transitions apply γ. Time-limit truncation retains bootstrap value from
the final observation, while true termination zeros it. The final row has no outgoing
training transition. Group boundaries prevent replay episodes from mixing.

With `next_step`, both components get the environment reward and bootstrap from the next
state's first readout. With `next_action`, the first component targets the second
component's value with no reward or discount. With `n_actions`, each target skips
exactly `ACTION_COMPONENTS` decisions, subject to the available replay group.


In [ ]:
gates = {
    "next_action": None,
    "n_actions": nstep_gate(n=ACTION_COMPONENTS),
    "next_step": nstep_gate(n=1, step_key="environment_step"),
}
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=0.99, gamma_terminated=0.0, gamma_truncated=1.0,
        step_key="environment_step",
    ),
    temperature=0.0,
    cross_group_backups="bootstrap",
    gate=gates[BACKUP_MODE],
)
optimizer = AdamW(
    params=model.parameter_groups(
        lr={
            "embedding": LR_EMBEDDING,
            "backbone": LR_BACKBONE,
            "projection": LR_PROJECTION,
            "value_head": LR_HEADS,
        },
    ),
    lr=LR_BACKBONE,
    weight_decay=0.0,
)


def run_train(*, store, cycle, num_steps):
    loader = DataLoader(
        stores=[store],
        samples_budget=(MAX_STEPS_PER_EPISODE * ACTION_COMPONENTS + 1) * EPISODES_PER_CYCLE,
        batch_size=1,
        sample_field="episode_index",
        transform=tokenizer,
        num_workers=0,
        seed=SEED + cycle,
    )
    try:
        model.train()
        for _ in range(num_steps):
            inputs, objective_data, group_id = loader.next_batch()
            out = model(inputs)
            with torch.no_grad():
                delayed_out = delayed_model(inputs)
            losses, metrics = objective(
                objective_data=to_device(data=objective_data, device=device),
                group_id=group_id.to(device),
                predictions=out["predictions"]["action_value"],
                backup_value_predictions=delayed_out["predictions"]["backup_value"],
                backup_action_predictions=delayed_out["predictions"]["backup_action"],
            )
            loss = torch.stack(tuple(losses.values())).sum()
            assert torch.isfinite(loss)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
            optimizer.step()
            delayed_model.polyak(
                online=model,
                embedding_tau=POLYAK_TAU_EMBEDDING,
                backbone_tau=POLYAK_TAU_BACKBONE,
                projection_tau=POLYAK_TAU_PROJECTION,
                value_tau=POLYAK_TAU_VALUE,
                action_tau=POLYAK_TAU_ACTION,
            )
        loss_value = float(loss.detach())
        return loss_value, metrics
    finally:
        loader.close()

## Run

Collect complete episodes and train from replay.


In [ ]:
store = Datastore(name="sequential-reacher-replay")
training_log = []
try:
    for cycle in range(NUM_CYCLES):
        returns = []
        for episode in range(EPISODES_PER_CYCLE):
            episode_return, last_history = collect_episode(
                episode_index=cycle * EPISODES_PER_CYCLE + episode, store=store,
            )
            returns.append(episode_return)
        loss, metrics = run_train(store=store, cycle=cycle, num_steps=TRAIN_STEPS)
        training_log.append({"cycle": cycle, "mean_return": float(np.mean(returns)), "loss": loss})
        print(
            f"cycle={cycle} | mean return={np.mean(returns):.1f} | "
            f"loss={loss:.4f} | mean Q={float(metrics['q_values_mean']):.3f} | replay rows={len(store)}"
        )
finally:
    env.close()

print(f"Finished {NUM_CYCLES * EPISODES_PER_CYCLE} episodes and {NUM_CYCLES * TRAIN_STEPS} updates.")

## Check the decision and environment clocks

There are K decisions per physical transition and one final bootstrap row. Changing an
objective-only ID never changes the tokens; later choices instead see the recorded torque.
The checks below exercise the actual packed pipeline and the three backup settings.


In [ ]:
assert len(last_history) == MAX_STEPS_PER_EPISODE * ACTION_COMPONENTS + 1
for index, row in enumerate(last_history[1:], start=1):
    assert row["environment_step"] == (index % ACTION_COMPONENTS == 0)
    if not row["environment_step"]:
        assert row["reward"] == 0 and not row["terminated"] and not row["truncated"]
        np.testing.assert_array_equal(row["observation"], last_history[index - 1]["observation"])
    assert row["torque"] == float(head.discrete_to_continuous(row["action_id"])[0])
print(f"Verified {len(last_history) - 1} decisions and "
      f"{sum(row['environment_step'] for row in last_history)} environment transitions.")
